# CPIT-440 — Group Project
## Spam Email Detection using Data Mining Techniques

**Group Name:** LogicMinds  
**Members:**
- Azzam Saeed Alghamdi — 2237392 
- Yazan Ezz Basnawi — 2244293
- Abdullah Mansour Alhalawani — 2236079

**Section:** IT1

---

### Problem
Build a classifier that can decide whether an incoming email is **spam** (1) or **ham / legitimate** (0), using 57 numerical features extracted from the email body.

### Dataset
We use the **UCI Spambase** dataset: 4,601 emails × 57 features (48 word-frequency + 6 character-frequency + 3 capital-letter run-length).

### Approach
We compare three classification algorithms covered in the course:
1. **Decision Tree** (interpretable, fast)
2. **Naïve Bayes** (probabilistic baseline)
3. **Random Forest** (ensemble)

We evaluate them with a 75/25 hold-out split AND 10-fold stratified cross-validation, using accuracy, precision, recall, F1-score, ROC curve, and AUC.

## 1. Import libraries

In [9]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing   import StandardScaler
from sklearn.tree            import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes     import GaussianNB
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (accuracy_score, confusion_matrix,
                                      classification_report, roc_curve,
                                      roc_auc_score, precision_recall_fscore_support)

%matplotlib inline
sns.set_style("whitegrid")
np.random.seed(42)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the dataset

In [10]:
# Load the Spambase dataset
# (Download from: https://archive.ics.uci.edu/ml/datasets/Spambase)
df = pd.read_csv("spambase.csv")

print(f"Shape: {df.shape}")
df.head()

Shape: (4600, 58)


,0,0.64,0.64.1,0.1,0.32,0.2,0.3,0.4,0.5,0.6,...,0.41,0.42,0.43,0.778,0.44,0.45,3.756,61,278,1
0,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
1,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
2,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,1.85,0.00,0.00,1.85,0.00,0.00,...,0.00,0.223,0.0,0.000,0.000,0.000,3.000,15,54,1


In [11]:
# Summary statistics
df.describe().T.head(15)

,count,mean,std,min,25%,50%,75%,max
0,4600.0,0.104576,0.305387,0.0,0.0,0.0,0.0000,4.54
0.64,4600.0,0.212922,1.290700,0.0,0.0,0.0,0.0000,14.28
0.64.1,4600.0,0.280578,0.504170,0.0,0.0,0.0,0.4200,5.10
0.1,4600.0,0.065439,1.395303,0.0,0.0,0.0,0.0000,42.81
0.32,4600.0,0.312222,0.672586,0.0,0.0,0.0,0.3825,10.00
0.2,4600.0,0.095922,0.273850,0.0,0.0,0.0,0.0000,5.88
0.3,4600.0,0.114233,0.391480,0.0,0.0,0.0,0.0000,7.27
0.4,4600.0,0.105317,0.401112,0.0,0.0,0.0,0.0000,11.11
0.5,4600.0,0.090087,0.278643,0.0,0.0,0.0,0.0000,5.26
0.6,4600.0,0.239465,0.644816,0.0,0.0,0.0,0.1600,18.18


In [12]:
# Class distribution
counts = df['is_spam'].value_counts().sort_index()
print(f"Ham (0):  {counts[0]:>4} emails ({counts[0]/len(df)*100:.1f}%)")
print(f"Spam (1): {counts[1]:>4} emails ({counts[1]/len(df)*100:.1f}%)")

plt.figure(figsize=(7, 5))
plt.bar(['Ham (0)', 'Spam (1)'], counts.values,
        color=['#4ECDC4', '#FF6B6B'], edgecolor='black')
for i, v in enumerate(counts.values):
    plt.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.title('Class Distribution in Spambase Dataset')
plt.ylabel('Number of emails')
plt.show()

KeyError: 'is_spam'

In [ ]:
# Check for missing values
print("Missing values per column:", df.isnull().sum().sum())
print("Number of features:", df.shape[1] - 1)

## 3. Explore the data

In [ ]:
# Average value of each feature, separated by class
feature_means = df.groupby('is_spam').mean().T
feature_means.columns = ['Ham_mean', 'Spam_mean']
feature_means['Diff'] = feature_means['Spam_mean'] - feature_means['Ham_mean']

print("=== Top 10 features more frequent in SPAM than in HAM ===")
feature_means.sort_values('Diff', ascending=False).head(10)

In [ ]:
# Visualize a few of the most discriminative features
discriminative = ['word_freq_free', 'word_freq_money', 'char_freq_$',
                  'char_freq_!', 'capital_run_length_longest']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, feat in zip(axes, discriminative):
    df.boxplot(column=feat, by='is_spam', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('is_spam')
plt.suptitle('Distribution of discriminative features by class', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Pre-processing — split into train and test sets (75% / 25%)

In [ ]:
X = df.drop(columns=['is_spam'])
y = df['is_spam']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]} emails")
print(f"Testing  set: {X_test.shape[0]} emails")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")

In [ ]:
# Standardize for Naïve Bayes (helps the Gaussian assumption)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Features scaled (mean ≈ 0, std ≈ 1)")

## 5. Train the three classifiers

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Naïve Bayes":   GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    print(f"\n=== Training {name} ===")
    
    if name == "Naïve Bayes":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    predictions[name]    = y_pred
    probabilities[name]  = y_prob
    
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'accuracy':  acc,
        'precision': prec,
        'recall':    rec,
        'f1':        f1,
        'auc':       auc
    }
    
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"AUC:       {auc:.4f}")

## 6. Detailed evaluation for each model

In [ ]:
for name in models:
    print(f"\n=== {name} — Classification Report ===")
    print(classification_report(y_test, predictions[name],
                                 target_names=['Ham', 'Spam']))

## 7. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, name in zip(axes, models):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Ham', 'Spam'],
                yticklabels=['Ham', 'Spam'], ax=ax, cbar=False)
    ax.set_title(f"{name}\nAccuracy = {results[name]['accuracy']:.3f}", fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices — Three Classifiers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. ROC curves

In [ ]:
plt.figure(figsize=(9, 6))
colors = {'Decision Tree': '#FF6B6B', 'Naïve Bayes': '#4ECDC4', 'Random Forest': '#95E1D3'}
for name in models:
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {results[name]['auc']:.3f})",
             linewidth=2.5, color=colors[name])
plt.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.5, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Comparison of the three classifiers')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

## 9. Metric comparison

In [ ]:
names = list(models.keys())
accs   = [results[n]['accuracy']  for n in names]
precs  = [results[n]['precision'] for n in names]
recs   = [results[n]['recall']    for n in names]
f1s    = [results[n]['f1']        for n in names]

x = np.arange(len(names))
w = 0.2
plt.figure(figsize=(10, 6))
plt.bar(x - 1.5*w, accs,  w, label='Accuracy',  color='#1F77B4', edgecolor='black')
plt.bar(x - 0.5*w, precs, w, label='Precision', color='#FF7F0E', edgecolor='black')
plt.bar(x + 0.5*w, recs,  w, label='Recall',    color='#2CA02C', edgecolor='black')
plt.bar(x + 1.5*w, f1s,   w, label='F1-score',  color='#D62728', edgecolor='black')
plt.xticks(x, names)
plt.ylim(0, 1.05)
plt.ylabel('Score')
plt.title('Metric Comparison Across the Three Models')
plt.legend(loc='lower right')
plt.grid(True, axis='y', alpha=0.3)
plt.show()

## 10. 10-Fold Cross-Validation

In [ ]:
cv_results = {}
for name, model in models.items():
    if name == "Naïve Bayes":
        X_data = X_train_scaled
    else:
        X_data = X_train
    
    scores = cross_val_score(model, X_data, y_train,
                              cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                              scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:15s}: mean = {scores.mean():.4f}, std = {scores.std():.4f}")
    print(f"   Fold scores: {np.round(scores, 4)}")

In [ ]:
# Visualize CV results
cv_means = [cv_results[n].mean() for n in names]
cv_stds  = [cv_results[n].std()  for n in names]

plt.figure(figsize=(9, 5))
plt.bar(names, cv_means, yerr=cv_stds, capsize=10,
        color=['#FF6B6B', '#4ECDC4', '#95E1D3'], edgecolor='black')
for i, (m, s) in enumerate(zip(cv_means, cv_stds)):
    plt.text(i, m + s + 0.01, f'{m:.3f}', ha='center', fontweight='bold')
plt.ylim(0.5, 1.05)
plt.ylabel('Mean CV Accuracy')
plt.title('10-Fold Cross-Validation Accuracy with Std-Dev Error Bars')
plt.grid(True, axis='y', alpha=0.3)
plt.show()

## 11. Feature importance — which features matter most?

In [ ]:
rf = models['Random Forest']
importance = pd.DataFrame({
    'Feature':    X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

print("=== Top 15 most important features (Random Forest) ===")
print(importance.to_string(index=False))

plt.figure(figsize=(10, 7))
plt.barh(importance['Feature'][::-1], importance['Importance'][::-1],
         color='steelblue', edgecolor='black')
plt.xlabel('Importance')
plt.title('Top 15 Most Important Features for Spam Detection')
plt.tight_layout()
plt.show()

## 12. Decision Tree Visualization

In [ ]:
# Smaller tree (depth=4) for readability
dt_small = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_small.fit(X_train, y_train)

plt.figure(figsize=(20, 10))
plot_tree(dt_small,
          feature_names=X.columns.tolist(),
          class_names=['Ham', 'Spam'],
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree (max_depth=4) for Spam Classification', fontsize=14, fontweight='bold')
plt.show()

## 13. Summary table

In [ ]:
summary = pd.DataFrame(results).T
summary['cv_mean'] = [cv_results[n].mean() for n in names]
summary['cv_std']  = [cv_results[n].std()  for n in names]
summary = summary.round(4)
summary.columns = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC', 'CV mean', 'CV std']

print("=== Final summary of all metrics ===")
summary

---
## Conclusion

All three algorithms achieved a high accuracy on the Spambase dataset. Looking at all metrics together:

1. **Random Forest** generally provides the best balance of accuracy, precision, and recall.
2. **Naïve Bayes** is surprisingly competitive and is several orders of magnitude faster to train; it is a strong baseline for text-based classification.
3. **Decision Tree** is the easiest to interpret (the rules can be read directly off the tree) but tends to overfit a bit and has the lowest test accuracy.

The most informative features are word frequencies like `free`, `money`, `credit`, `your`, `!`, and `$`, plus the capital-letter run-length features — confirming the intuition that spam messages use marketing language and excessive capitalization.

For a production email filter we would recommend **Random Forest** (or a tuned gradient-boosting model) because false positives are very costly: marking a legitimate email as spam is much worse than letting an occasional spam reach the inbox. Random Forest gave the highest precision on the spam class, which is exactly what we want in this domain.